# Real-Time Face Mask Compliance Detection

To use the CDS6334 environment, run below conda command:

In [67]:
conda env export -n CDS6334 > CDS6334.yml


Note: you may need to restart the kernel to use updated packages.


In [2]:
# import libraries
# !pip install tensorflow opencv-python matplotlib seaborn scikit-learn
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

print("Libraries installed and imported successfully.")

Libraries installed and imported successfully.


## 1.0 Data Loading

### 1.1 Define Paths

In [69]:
DATASET_PATH = r"C:\Users\Lee Hong Yi\Downloads\VIP Assignment\Real-Time-Face-Mask-Detection\Dataset"
class_dirs = ["with_mask", "without_mask", "mask_weared_incorrect"]
print("DATASET_PATH exists:", os.path.exists(DATASET_PATH))
for d in class_dirs:
    p = os.path.join(DATASET_PATH, d)
    print(d, "exists:", os.path.exists(p), "num_images:", len(glob(os.path.join(p, "*.*"))))

DATASET_PATH exists: True
with_mask exists: True num_images: 2994
without_mask exists: True num_images: 2994
mask_weared_incorrect exists: True num_images: 2994


## 2.0 Data Preprocessing

### 2.1 Visual Sanity Check

In [ ]:
# picks one random image from each of the three folders and displays it 
# on the screen. 
# to ensure the data isn't corrupted and to visually confirm what 
# "Incorrect Mask" looks like. It helps you verify that the dataset is 
# actually cropped faces, not full-body images.

### 2.2 Configuration and Data Augmentation

In [ ]:
# This sets up the ImageDataGenerator. This is a tool that:
# 1. Rescales: Converts pixel colors from 0-255 to 0-1 (Neural networks like small numbers).
# 2. Splits: Automatically reserves 20% of the data for testing (Validation).
# 3. Augments: Defines rules to randomly rotate, zoom, or shift images during training. Why it is needed: Augmentation is critical for your project because "Incorrect Mask" detection is subtle. By randomly tilting images, we teach the model to recognize a nose slipping out even if the person's head isn't perfectly straight.

### 2.3 Create Train and Validation Generators

In [ ]:
# This actually connects the datagen (the rules) to your DATASET_PATH 
# (the files). It creates two "streams" of data: train_gen and val_gen. 
# These two variables (train_gen and val_gen) are what you will feed 
# directly into the model training command (model.fit) in Section 3.0.

### 2.2 Preprocessing for Model B (EfficientNet Format)

#### 2.2.1 Create Crop Directories

In [70]:
# Creates 3 folders: crops/with_mask, crops/without_mask, crops/incorrect.
# Ensure crop directories exist
CROPS_PATH = os.path.join(DATASET_PATH, "crops")
CROP_CLASSES = ["with_mask", "without_mask", "mask_weared_incorrect"]

for c in CROP_CLASSES:
    os.makedirs(os.path.join(CROPS_PATH, c), exist_ok=True)

def is_image_valid(img_path):
    try:
        img = cv2.imread(img_path)
        if img is None or img.size == 0:
            return False
        return True
    except Exception:
        return False

#### 2.2.2 Crop and Sort

In [71]:


# Logic: It loops through every image, looks at the bounding box, cuts only the face out of the image, 
# and saves that tiny face image into the correct folder.

# Why: EfficientNet is a classifier. It needs to look at just the face to decide if the mask is 
# correct or not.
def augment_image(img):
    aug_imgs = [img]
    if img is not None and img.size > 0:
        # Horizontal flip
        aug_imgs.append(cv2.flip(img, 1))
        # Rotate 15 degrees
        M = cv2.getRotationMatrix2D((img.shape[1]//2, img.shape[0]//2), 15, 1)
        aug_imgs.append(cv2.warpAffine(img, M, (img.shape[1], img.shape[0])))
        # Brightness adjustment
        aug_imgs.append(cv2.convertScaleAbs(img, alpha=1.2, beta=30))
    return aug_imgs

# Loop through all PNG images in each class folder
for c in CROP_CLASSES:
    img_dir = os.path.join(DATASET_PATH, c)
    img_files = glob(os.path.join(img_dir, "*.png"))
    for img_path in tqdm(img_files, desc=f"Processing {c}"):
        if not is_image_valid(img_path):
            continue  # Skip corrupted images
        img = cv2.imread(img_path)
        # If bounding box info is available, crop face here (not implemented, so use full image)
        face_img = img  # Placeholder for cropped face
        for idx, aug in enumerate(augment_image(face_img)):
            save_path = os.path.join(CROPS_PATH, c, f"{os.path.splitext(os.path.basename(img_path))[0]}_{idx}.png")
            cv2.imwrite(save_path, aug)

Processing with_mask:   0%|          | 0/2994 [00:00<?, ?it/s]

Processing with_mask:   0%|          | 0/2994 [00:00<?, ?it/s]

Processing mask_weared_incorrect: 100%|██████████| 2994/2994 [00:19<00:00, 155.58it/s]


In [72]:
# Optional: Preprocess to brighten and sharpen dim or blurry images before augmentation
# This will brighten and sharpen all images before saving to crops directory

def brighten_image(img, alpha=1.0, beta=40):
    # alpha: contrast (1.0-3.0), beta: brightness (0-100)
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

def sharpen_image(img):
    kernel = np.array([[0, -1, 0],
                      [-1, 5, -1],
                      [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)

for c in CROP_CLASSES:
    img_dir = os.path.join(DATASET_PATH, c)
    img_files = glob(os.path.join(img_dir, "*.png"))
    for img_path in tqdm(img_files, desc=f"Brightening & Sharpening {c}"):
        if not is_image_valid(img_path):
            continue
        img = cv2.imread(img_path)
        bright_img = brighten_image(img, alpha=1.0, beta=40)  # Adjust beta as needed
        sharp_img = sharpen_image(bright_img)
        # Use sharp_img for augmentation and saving
        face_img = sharp_img  # If bounding box, crop here
        for idx, aug in enumerate(augment_image(face_img)):
            save_path = os.path.join(CROPS_PATH, c, f"{os.path.splitext(os.path.basename(img_path))[0]}_{idx}.png")
            cv2.imwrite(save_path, aug)


Brightening & Sharpening mask_weared_incorrect: 100%|██████████| 2994/2994 [00:18<00:00, 163.96it/s]


#### 2.2.3 Data Generators

In [ ]:
# Sets up ImageDataGenerator which automatically loads these cropped images in batches and applies 
# "Augmentation" (randomly rotating them slightly) to make the model smarter.
# Paths
train_dir = os.path.join(CROPS_PATH)  # If you have a split, use subfolders like 'train', 'test'

# Data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2  # 20% for validation
)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)
print(f"[INFO] Training generator: Found {train_gen.samples} images belonging to {train_gen.num_classes} classes. These will be used for training.")

test_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)
print(f"[INFO] Testing generator: Found {test_gen.samples} images belonging to {test_gen.num_classes} classes. These will be used for Testing.")

Found 28743 images belonging to 3 classes.
[INFO] Training generator: Found 28743 images belonging to 3 classes. These will be used for training.
Found 7185 images belonging to 3 classes.
[INFO] Testing generator: Found 7185 images belonging to 3 classes. These will be used for validation.


## 3.0 Model A: EfficientNet-B0

### 3.1 Build Architecture

In [ ]:
# Downloads the pre-trained EfficientNet-B0 (trained on ImageNet). It removes the top layer (the one that classifies cats/dogs) and adds a Custom Head for your 3 mask classes.
# Transfer Learning saves you from training from scratch. EfficientNet uses "Compound Scaling" which is great for finding subtle details like a nose slipping out.

### 3.2 Train Model

In [ ]:
# Feeds the training images into the model for 15 epochs.
# epochs=15 is usually enough for transfer learning. You will see the accuracy go up and loss go down.

## 4.0 Model B: MobileNetV2

### 4.1 Build and Train

In [ ]:
# Performs the exact same steps as Section 3.0, but swaps the "Brain" for MobileNetV2.
# To ensure a fair comparison, we use the exact same data, same optimizer, and same output layers. The only variable changing is the architecture.

## 5.0 Comparative Evaluation

### 5.1 Training History Plots

In [ ]:
# Draws line graphs comparing how both models learned.
# To check if one model learned faster or if one model "overfit" (memorized the data instead of learning).

### 5.2 Confusion Matrix & Classification Report

In [ ]:
# Generates the "Report Card". It tells you exactly which classes were confused.
# Pay attention to the F1-Score for mask_weared_incorrect.
# This proves which model is safer. If EfficientNet has fewer False Negatives on incorrect masks, it wins on "Safety".

### 5.3 Inference Speed Test (FPS)

In [ ]:
# Runs 100 dummy images through both models and times them.
# This is the key metric for MobileNet. You expect MobileNet to be faster. This data point goes into your "Results" table.

## 6.0 Application Simulation (Access Control)